In [1]:
import os
import io
import time
import base64
from pathlib import Path
from getpass import getpass

import fitz
import pandas as pd

from PIL import Image
from IPython.display import display

from groq import Groq

from pinecone import Pinecone, ServerlessSpec

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

c:\Users\ce\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

print("GROQ_API_KEY configured:", bool(groq_api_key))
print("PINECONE_API_KEY configured:", bool(pinecone_api_key))

GROQ_API_KEY configured: True
PINECONE_API_KEY configured: True


In [4]:
from pathlib import Path

PDF_PATH = Path(r"P:\ADVANCED_RAG\Multimodal_RAG\NovaCore_Multimodal_Company_Report_2026.pdf")

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}")

print("PDF:", PDF_PATH)

PDF: P:\ADVANCED_RAG\Multimodal_RAG\NovaCore_Multimodal_Company_Report_2026.pdf


In [5]:
TEXT_MODEL = "openai/gpt-oss-20b"
VISION_MODEL = "qwen/qwen3.8-27b"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

PINECONE_INDEX_NAME = "multimodal-rag"
PINECONE_NAMESPACE = "fy2026-demo"

IMAGE_DIR = Path("extracted_images")
IMAGE_DIR.mkdir(exist_ok=True)

print("PDF:", PDF_PATH)
print("Pinecone index:", PINECONE_INDEX_NAME)
print("Pinecone namespace:", PINECONE_NAMESPACE)

PDF: P:\ADVANCED_RAG\Multimodal_RAG\NovaCore_Multimodal_Company_Report_2026.pdf
Pinecone index: multimodal-rag
Pinecone namespace: fy2026-demo


In [10]:
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

text_llm = ChatGroq(
    model=TEXT_MODEL,
    temperature=0,
)

# 5. Helper: convert image to base64

In [11]:
def image_to_data_uri(image_path):
    image_path = Path(image_path)

    with Image.open(image_path) as img:
        img = img.convert("RGB")
        img.thumbnail((1600, 1600))

        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=85)

    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{encoded}"

# 6. Helper: summarize images, charts, and diagrams


We do not embed raw images with the text embedding model.

Instead:

```text
Image / Chart
     ↓
Vision model
     ↓
Text summary
     ↓
Embedding
     ↓
Pinecone
```

We still keep the original image path so the final answer can use the real visual.

In [12]:
def summarize_visual(image_path, page_number):
    image_data = image_to_data_uri(image_path)

    prompt = f"""
This visual was extracted from page {page_number} of the NovaCore FY2026 company report.

Describe the useful business information visible in the visual.

If it is a chart or graph:
- mention important values
- mention highest/lowest values
- mention the main trend

If it is a diagram:
- identify important components
- explain the flow or relationships

If it is a normal business image:
- describe the useful factual information

Keep the description concise and factual.
"""

    response = groq_client.chat.completions.create(
        model=VISION_MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {"url": image_data},
                    },
                ],
            }
        ],
        temperature=0,
        max_completion_tokens=500,
    )

    return response.choices[0].message.content.strip()

# 7. Extract text, tables, and visuals from the PDF

In [13]:
def extract_multimodal_documents(pdf_path):
    pdf = fitz.open(str(pdf_path))

    documents = []
    extracted_xrefs = set()

    for page_index in range(len(pdf)):
        page = pdf[page_index]
        page_number = page_index + 1

        print(f"Processing page {page_number}...")

        # -------------------------------------------------
        # 1. TEXT
        # -------------------------------------------------
        text = page.get_text("text").strip()

        if text:
            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "page": page_number,
                        "modality": "text",
                        "source": pdf_path.name,
                    },
                )
            )

        # -------------------------------------------------
        # 2. TABLES
        # -------------------------------------------------
        try:
            tables = page.find_tables().tables

            for table_number, table in enumerate(tables, start=1):
                df = table.to_pandas()

                if not df.empty:
                    table_text = df.to_markdown(index=False)

                    documents.append(
                        Document(
                            page_content=table_text,
                            metadata={
                                "page": page_number,
                                "modality": "table",
                                "table_number": table_number,
                                "source": pdf_path.name,
                            },
                        )
                    )

        except Exception as error:
            print("Table extraction warning:", error)

        # -------------------------------------------------
        # 3. IMAGES / CHARTS / DIAGRAMS
        # -------------------------------------------------
        for image_number, image_info in enumerate(
            page.get_images(full=True),
            start=1,
        ):
            xref = image_info[0]

            # Do not process an identical embedded image twice.
            if xref in extracted_xrefs:
                continue

            extracted_xrefs.add(xref)

            image_data = pdf.extract_image(xref)
            image_bytes = image_data["image"]
            image_extension = image_data["ext"]

            image_path = (
                IMAGE_DIR
                / f"page_{page_number}_image_{image_number}.{image_extension}"
            )

            image_path.write_bytes(image_bytes)

            try:
                summary = summarize_visual(
                    image_path=image_path,
                    page_number=page_number,
                )

                documents.append(
                    Document(
                        page_content=summary,
                        metadata={
                            "page": page_number,
                            "modality": "visual",
                            "image_path": str(image_path),
                            "source": pdf_path.name,
                        },
                    )
                )

            except Exception as error:
                print(
                    f"Vision warning on page {page_number}:",
                    error,
                )

    pdf.close()
    return documents


documents = extract_multimodal_documents(PDF_PATH)

print()
print("Total LangChain documents:", len(documents))

Processing page 1...
Processing page 2...
Processing page 3...
Processing page 4...
Processing page 5...
Processing page 6...
Processing page 7...
Processing page 8...
Processing page 9...

Total LangChain documents: 24


In [14]:
documents

[Document(metadata={'page': 1, 'modality': 'text', 'source': 'NovaCore_Multimodal_Company_Report_2026.pdf'}, page_content='NovaCore Systems Ltd. - Fictional company report created for a multimodal RAG demonstration\nPage 1\nNOVACORE SYSTEMS LTD.\nFY2026 BUSINESS PERFORMANCE &\nOPERATIONS REPORT\nA realistic fictional company document designed for multimodal RAG testing\nHeadquarters\nSingapore\nEmployees\n1,480\n2026 Revenue\n$132.0M\nOperating Margin\n18.4%\nPrimary Sector\nIndustrial AI & IoT\nMarkets Served\n42 countries\nDemo note: All company names, metrics, people, and events in this report are fictional. The document intentionally mixes text, tables,\ncharts, and embedded images so it can be used to demonstrate multimodal retrieval and question answering.'),
 Document(metadata={'page': 1, 'modality': 'table', 'table_number': 1, 'source': 'NovaCore_Multimodal_Company_Report_2026.pdf'}, page_content='| Headquarters   | Singapore           | Employees        | 1,480        |\n|:---